<a href="https://colab.research.google.com/github/sonmezo21/GeoAI-Wildfire-Triage/blob/main/MYZ305E_GeoAI_Projesi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Gerekli tüm kütüphanelerin en güncel versiyonlarını kuruyoruz
!pip install -U geopandas google-generativeai langchain-google-genai langchain langchain-experimental

In [ ]:
# 2. Drive'ı bağlama ve veriyi okuma
from google.colab import drive
import geopandas as gpd
from IPython.display import display

drive.mount('/content/drive')

# 2025 Ege-Akdeniz Yangınları verimizi okuyoruz
dosya_yolu = "/content/drive/MyDrive/GeoAI_Proje/Ege_Akdeniz_Yanginlari_2025_Temiz.shp"
yangin_verisi = gpd.read_file(dosya_yolu, encoding='latin1')

print("Harita verisi başarıyla yüklendi! İlk 3 satır:")
display(yangin_verisi.head(3))

In [ ]:
# 3. Güvenli API Kimlik Doğrulaması
import os
from getpass import getpass
import google.generativeai as genai

# Kodu çalıştıran her yeni kişiye kendi anahtarını sorar, kimsenin kotası karışmaz
print("Lütfen Google Gemini API anahtarınızı yapıştırın ve Enter'a basın:")
os.environ["GOOGLE_API_KEY"] = getpass()

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
print("\nAPI Anahtarı onaylandı. Sistem devrede!")

In [ ]:
# 4. GeoAI Ajanını oluşturma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Ajanı coğrafi verimizle birleştiriyoruz
ajan = create_pandas_dataframe_agent(
    llm,
    yangin_verisi,
    verbose=True,
    allow_dangerous_code=True,
    handle_parsing_errors=True
)

print("\nOtonom GeoAI Ajanı operasyona hazır! 🚀 Görev bekliyor.")

In [ ]:
# 5. Sistem Sağlık Kontrolü ve Model Doğrulaması
import google.generativeai as genai

print("Available Models and their supported methods:")
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(f"  Model: {m.name}, Supported Methods: {m.supported_generation_methods}")

In [ ]:
# 6. Ajana operasyonel görevini (Prompt) veriyoruz: Mekansal Triyaj ve Çubuk Grafik
gorev = """
Analyze the provided GeoDataFrame and perform the following tasks:
1. Identify the column representing the province or city names (e.g., 'IL_ADI', 'NAME', 'PROVINCE').
2. Calculate the total burned area for each province.
3. Find the Top 5 provinces with the largest total burned area.
4. Generate a bar chart showing these Top 5 provinces.
5. Use 'darkred' as the color for the bars.
6. Set the title of the chart exactly as 'Top 5 Provinces by Total Burned Area'. Label the x and y axes appropriately in English.
"""

print("Ajan görevi aldı, mekansal triyajı yapıyor ve grafiği çiziyor... (Bu işlem biraz sürebilir)\n")

# Ajanı çalıştırıyoruz
yanit = ajan.invoke(gorev)

print("\n--- Otonom Ajanın Raporu ---")
print(yanit["output"])

In [ ]:
# 7. Ajanımıza daha gelişmiş bir Afet Triyaj Haritası yaptırıyoruz
gelismis_gorev = """
Sen bir GeoAI Afet Müdahale uzmanısın. 'yangin_verisi' isimli GeoDataFrame üzerinde şu işlemleri yap:

1. Her poligonun alanını hesapla ve veriyi alana göre büyükten küçüğe sırala.
2. Folium kullanarak Ege ve Akdeniz bölgesini gösterecek bir altlık harita oluştur.
3. Alanı EN BÜYÜK 5 YANGINI (Top 5) tespit et. Bu 5 yangının merkez koordinatlarına kırmızı birer Marker ekle ve her birinin etrafına afet müdahalesi için 5 kilometrelik (5000 metre) yarı saydam kırmızı çemberler (risk bölgesi) çiz.
4. Geriye kalan diğer tüm yangınların merkez koordinatlarına ise sadece küçük, sarı renkli CircleMarker'lar (yarıçapı 2 olsun) ekle.
5. Haritayı 'afet_triyaj_haritasi.html' adıyla kaydet.
6. Bana İngilizce olarak Top 5 yangının bulunduğu koordinatları ve oluşturduğun bu triyaj mantığını özetle.
"""

cevap_gelismis_harita = ajan.invoke(gelismis_gorev)

print("\n--- AJANIN İNGİLİZCE ÖZETİ ---")
print(cevap_gelismis_harita["output"])
